# 2D Lid-Driven Cavity CFD Solver

A 2D incompressible Navier-Stokes solver for the classic lid-driven cavity benchmark, implemented from scratch in Python using the **projection (fractional-step) method**. Validated against the widely-cited **Ghia et al. (1982)** benchmark data at Re = 100, 400, and 1000.

---

## Physical Setup

The lid-driven cavity is a square domain [0,1]×[0,1] filled with a viscous, incompressible fluid. The **top wall** moves at a constant velocity $u = 1$, while the other three walls are stationary. Despite the simple geometry, the flow develops a rich vortical structure whose exact solution is well-documented in the literature — making it ideal for solver validation.

```
        u = 1 (moving lid)
   ┌──────────────────────┐
   │                      │
u=0│    recirculating     │u=0
v=0│       flow           │v=0
   │                      │
   └──────────────────────┘
        u = 0, v = 0
```

## Governing Equations

The incompressible Navier-Stokes equations:

$$\frac{\partial \mathbf{u}}{\partial t} + (\mathbf{u} \cdot \nabla)\mathbf{u} = -\frac{1}{\rho}\nabla p + \nu \nabla^2 \mathbf{u}$$

$$\nabla \cdot \mathbf{u} = 0$$

## Numerical Method

The solver uses the **projection method** (Chorin 1968) to decouple velocity and pressure:

| Step | Name | Description |
|------|------|-------------|
| 1 | Predictor | Advance velocity explicitly using convection + diffusion (no pressure) → $\mathbf{u}^*$ |
| 2 | Pressure Poisson | Solve $\nabla^2 p = (\rho/\Delta t)\,\nabla \cdot \mathbf{u}^*$ via Jacobi iteration |
| 3 | Corrector | Project $\mathbf{u}^* $ onto divergence-free space: $\mathbf{u}^{n+1} = \mathbf{u}^* - (\Delta t/\rho)\nabla p$ |

Spatial discretisation uses **second-order central differences** on a uniform Cartesian grid. Time integration is **first-order explicit (forward Euler)**.


## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors


## 2. Grid Setup

A uniform Cartesian grid of $65 \times 65$ nodes covers the unit square $[0,1]^2$.

The array convention used throughout is:

```
u[i, j]  →  x-velocity at  x = x[j],  y = y[i]
```

Row 0 is the **bottom** wall ($y=0$) and row 64 is the **top** wall / lid ($y=1$).


In [ ]:
# Grid dimensions (match Ghia et al. 1982 resolution)
nx, ny = 65, 65       # nodes in x and y
Lx, Ly = 1.0, 1.0    # domain size

dx = Lx / (nx - 1)   # uniform grid spacing
dy = Ly / (ny - 1)

x = np.linspace(0.0, Lx, nx)
y = np.linspace(0.0, Ly, ny)
X, Y = np.meshgrid(x, y)   # X[i,j] = x[j],  Y[i,j] = y[i]

print(f"Grid : {nx} × {ny}")
print(f"dx   = dy = {dx:.6f}")


## 3. Boundary Conditions

### Velocity
| Wall | Condition |
|------|-----------|
| Top ($y=1$) | $u = 1$ (lid), $v = 0$ |
| Bottom, left, right | $u = v = 0$ (no-slip) |

### Pressure
| Wall | Condition |
|------|-----------|
| Top ($y=1$) | $p = 0$ (Dirichlet) |
| Bottom, left, right | $\partial p / \partial n = 0$ (Neumann) |

The Neumann condition is implemented with a **one-sided zero-gradient** extrapolation (e.g. `p[0, :] = p[1, :]` for the bottom wall).


In [ ]:
def apply_velocity_bc(u, v, u_lid=1.0):
    """Enforce velocity boundary conditions on all four walls."""
    # Bottom wall  (y = 0)  — no-slip
    u[0, :]  = 0.0;   v[0, :]  = 0.0
    # Top wall     (y = 1)  — moving lid
    u[-1, :] = u_lid; v[-1, :] = 0.0
    # Left wall    (x = 0)  — no-slip
    u[:, 0]  = 0.0;   v[:, 0]  = 0.0
    # Right wall   (x = 1)  — no-slip
    u[:, -1] = 0.0;   v[:, -1] = 0.0
    return u, v


def apply_pressure_bc(p):
    """Enforce pressure boundary conditions on all four walls."""
    p[-1, :] = 0.0       # top  : Dirichlet  p = 0
    p[0, :]  = p[1, :]   # bottom : dp/dn = 0
    p[:, 0]  = p[:, 1]   # left   : dp/dn = 0
    p[:, -1] = p[:, -2]  # right  : dp/dn = 0
    return p


## 4. Predictor Step — `vel_without_pressure`

Advance the velocity field by one explicit Euler step using **only** the convective and diffusive terms (no pressure gradient). The result $\mathbf{u}^*$ is in general **not** divergence-free.

$$u^*_{i,j} = u^n_{i,j} + \Delta t \left[ -u^n \frac{\partial u^n}{\partial x} -v^n \frac{\partial u^n}{\partial y} + \nu \left(\frac{\partial^2 u^n}{\partial x^2} + \frac{\partial^2 u^n}{\partial y^2}\right) \right]$$

All spatial derivatives are approximated with **central differences**:

$$\frac{\partial u}{\partial x} \approx \frac{u_{i,j+1} - u_{i,j-1}}{2\,\Delta x}, \qquad \frac{\partial^2 u}{\partial x^2} \approx \frac{u_{i,j+1} - 2u_{i,j} + u_{i,j-1}}{\Delta x^2}$$


In [ ]:
def vel_without_pressure(u, v, dt, dx, dy, nu):
    """
    Predictor step: advance velocity using convection and diffusion only.

    Uses central differences for all spatial derivatives.
    Boundary conditions are re-applied to the updated field.

    Parameters
    ----------
    u, v : ndarray (ny, nx)  current velocity components
    dt   : float             time-step
    dx, dy : float           grid spacing
    nu   : float             kinematic viscosity

    Returns
    -------
    u, v : ndarray  intermediate velocity field u*, v*
    """
    un, vn = u.copy(), v.copy()

    # ── x-momentum ─────────────────────────────────────────────────────────
    u[1:-1, 1:-1] = (
        un[1:-1, 1:-1]
        # convection  (central differences)
        - dt * un[1:-1, 1:-1] * (un[1:-1, 2:] - un[1:-1, :-2]) / (2 * dx)
        - dt * vn[1:-1, 1:-1] * (un[2:, 1:-1] - un[:-2, 1:-1]) / (2 * dy)
        # diffusion   (central differences)
        + nu * dt * (un[1:-1, 2:] - 2*un[1:-1, 1:-1] + un[1:-1, :-2]) / dx**2
        + nu * dt * (un[2:, 1:-1] - 2*un[1:-1, 1:-1] + un[:-2, 1:-1]) / dy**2
    )

    # ── y-momentum ─────────────────────────────────────────────────────────
    v[1:-1, 1:-1] = (
        vn[1:-1, 1:-1]
        # convection
        - dt * un[1:-1, 1:-1] * (vn[1:-1, 2:] - vn[1:-1, :-2]) / (2 * dx)
        - dt * vn[1:-1, 1:-1] * (vn[2:, 1:-1] - vn[:-2, 1:-1]) / (2 * dy)
        # diffusion
        + nu * dt * (vn[1:-1, 2:] - 2*vn[1:-1, 1:-1] + vn[1:-1, :-2]) / dx**2
        + nu * dt * (vn[2:, 1:-1] - 2*vn[1:-1, 1:-1] + vn[:-2, 1:-1]) / dy**2
    )

    u, v = apply_velocity_bc(u, v)
    return u, v


## 5. Pressure Poisson Equation — `pressure_poisson`

Taking the divergence of the momentum equation and enforcing incompressibility gives the **pressure Poisson equation**:

$$\nabla^2 p = \frac{\rho}{\Delta t} \nabla \cdot \mathbf{u}^*$$

The right-hand side $b_{i,j}$ is:

$$b_{i,j} = \frac{\rho}{\Delta t} \left(\frac{u^*_{i,j+1} - u^*_{i,j-1}}{2\,\Delta x} + \frac{v^*_{i+1,j} - v^*_{i-1,j}}{2\,\Delta y}\right)$$

The discretised Laplacian is inverted with **Jacobi iteration**:

$$p^{k+1}_{i,j} = \frac{\Delta y^2\,(p^k_{i,j+1}+p^k_{i,j-1}) + \Delta x^2\,(p^k_{i+1,j}+p^k_{i-1,j})}{2(\Delta x^2+\Delta y^2)} - \frac{\Delta x^2\,\Delta y^2}{2(\Delta x^2+\Delta y^2)} b_{i,j}$$

200 iterations are performed per time step — sufficient for the pressure to track the velocity divergence as it diminishes toward steady state.


In [ ]:
def pressure_poisson(p, u, v, dt, dx, dy, rho=1.0, niter=200):
    """
    Solve the pressure Poisson equation using Jacobi iteration.

    Parameters
    ----------
    p         : ndarray (ny, nx)  pressure field (updated in-place style)
    u, v      : ndarray (ny, nx)  intermediate velocity u*, v*
    dt        : float             time-step
    dx, dy    : float             grid spacing
    rho       : float             fluid density (default 1.0)
    niter     : int               number of Jacobi iterations (default 200)

    Returns
    -------
    p : ndarray  updated pressure field
    """
    # Build the RHS: divergence of intermediate velocity
    b = np.zeros_like(p)
    b[1:-1, 1:-1] = (
        rho / dt * (
            (u[1:-1, 2:] - u[1:-1, :-2]) / (2 * dx)
          + (v[2:, 1:-1] - v[:-2, 1:-1]) / (2 * dy)
        )
    )

    coeff = dx**2 * dy**2 / (2 * (dx**2 + dy**2))

    for _ in range(niter):
        pn = p.copy()
        p[1:-1, 1:-1] = (
            (dy**2 * (pn[1:-1, 2:]  + pn[1:-1, :-2])
           + dx**2 * (pn[2:,  1:-1] + pn[:-2,  1:-1]))
            / (2 * (dx**2 + dy**2))
            - coeff * b[1:-1, 1:-1]
        )
        p = apply_pressure_bc(p)

    return p


## 6. Corrector Step — `update_velocity`

Project $\mathbf{u}^*$ onto the space of divergence-free fields by subtracting the pressure gradient:

$$u^{n+1}_{i,j} = u^*_{i,j} - \frac{\Delta t}{\rho} \frac{p_{i,j+1} - p_{i,j-1}}{2\,\Delta x}$$

$$v^{n+1}_{i,j} = v^*_{i,j} - \frac{\Delta t}{\rho} \frac{p_{i+1,j} - p_{i-1,j}}{2\,\Delta y}$$

After correction the velocity field satisfies $\nabla \cdot \mathbf{u} \approx 0$ to within the Jacobi solver tolerance.


In [ ]:
def update_velocity(u, v, p, dt, dx, dy, rho=1.0):
    """
    Corrector step: subtract the pressure gradient from u*, v*.

    Only interior nodes are modified; boundary conditions are
    re-applied afterwards.
    """
    u[1:-1, 1:-1] -= dt / (rho * 2 * dx) * (p[1:-1, 2:]  - p[1:-1, :-2])
    v[1:-1, 1:-1] -= dt / (rho * 2 * dy) * (p[2:,  1:-1] - p[:-2,  1:-1])
    u, v = apply_velocity_bc(u, v)
    return u, v


## 7. Main Simulation Loop — `simulate_cavity_flow`

The three steps above are repeated until the flow reaches **steady state**. The maximum velocity divergence $|\nabla \cdot \mathbf{u}|_{\max}$ is printed every 500 steps as a convergence monitor; it should fall to $O(10^{-3})$ or smaller.

### Simulation parameters

| Re | $\nu$ | Steps | $\Delta t$ |
|----|--------|-------|-----------|
| 100  | 0.01   | 11 000 | 0.001  |
| 400  | 0.0025 | 30 000 | 0.001  |
| 1000 | 0.001  | 50 000 | 0.0005 |

The kinematic viscosity is computed from $\nu = U L / Re$ with $U=L=1$.


In [ ]:
def simulate_cavity_flow(Re, nsteps, dt,
                         niter_poisson=200, nx=65, ny=65,
                         rho=1.0, u_lid=1.0):
    """
    Run the lid-driven cavity flow to steady state.

    Parameters
    ----------
    Re            : int/float   Reynolds number
    nsteps        : int         number of time steps
    dt            : float       time-step size
    niter_poisson : int         Jacobi iterations per step (default 200)
    nx, ny        : int         grid dimensions (default 65×65)
    rho           : float       fluid density   (default 1.0)
    u_lid         : float       lid velocity    (default 1.0)

    Returns
    -------
    u, v, p : ndarray (ny, nx)  steady-state velocity and pressure fields
    """
    nu  = u_lid * 1.0 / Re       # kinematic viscosity  (L=1)
    _dx = 1.0 / (nx - 1)
    _dy = 1.0 / (ny - 1)

    u = np.zeros((ny, nx))
    v = np.zeros((ny, nx))
    p = np.zeros((ny, nx))
    u, v = apply_velocity_bc(u, v, u_lid)

    print(f"Re = {Re:4d}   nu = {nu:.5f}   dt = {dt}   steps = {nsteps}")
    print("-" * 60)

    for step in range(nsteps):
        # ── 1. Predictor ────────────────────────────────────────────
        u, v = vel_without_pressure(u, v, dt, _dx, _dy, nu)

        # ── 2. Pressure Poisson ─────────────────────────────────────
        p = pressure_poisson(p, u, v, dt, _dx, _dy, rho, niter_poisson)

        # ── 3. Corrector ─────────────────────────────────────────────
        u, v = update_velocity(u, v, p, dt, _dx, _dy, rho)

        # ── Convergence monitor ──────────────────────────────────────
        if (step + 1) % 500 == 0:
            div_max = np.max(np.abs(
                (u[1:-1, 2:] - u[1:-1, :-2]) / (2*_dx)
              + (v[2:, 1:-1] - v[:-2, 1:-1]) / (2*_dy)
            ))
            print(f"  step {step+1:6d}/{nsteps}   |∇·u|_max = {div_max:.3e}")

    print("Done.\n")
    return u, v, p


## 8. Ghia et al. (1982) Reference Data

Ghia, Ghia & Shin (1982) solved the cavity problem with a multigrid method on a fine grid and published benchmark velocity profiles. These are the de-facto reference for this test case.

- **Table 1** — $u$-velocity along the **vertical centreline** $x = 0.5$
- **Table 2** — $v$-velocity along the **horizontal centreline** $y = 0.5$

Each table gives 17 sample points at specific node locations.


In [ ]:
# ── u-velocity at the vertical centreline x = 0.5 ──────────────────────────
ghia_y = np.array([
    0.0000, 0.0547, 0.0625, 0.0703, 0.1016, 0.1719,
    0.2813, 0.4531, 0.5000, 0.6172, 0.7344, 0.8516,
    0.9531, 0.9609, 0.9688, 0.9766, 1.0000
])

ghia_u = {
    100: np.array([
         0.00000, -0.03717, -0.04192, -0.04775, -0.06434, -0.10150,
        -0.15662, -0.21090, -0.20581, -0.13641,  0.00332,  0.23151,
         0.68717,  0.73722,  0.78871,  0.84123,  1.00000]),
    400: np.array([
         0.00000, -0.08186, -0.09266, -0.10338, -0.14612, -0.24299,
        -0.32726, -0.17119, -0.11477,  0.02135,  0.16256,  0.29093,
         0.55892,  0.61756,  0.68439,  0.75837,  1.00000]),
    1000: np.array([
         0.00000, -0.18109, -0.20196, -0.22220, -0.29730, -0.38289,
        -0.27805, -0.10648, -0.06080,  0.05702,  0.18719,  0.33304,
         0.46604,  0.51117,  0.57492,  0.65928,  1.00000]),
}

# ── v-velocity at the horizontal centreline y = 0.5 ────────────────────────
ghia_x = np.array([
    0.0000, 0.0625, 0.0703, 0.0781, 0.0938, 0.1563,
    0.2266, 0.2344, 0.5000, 0.8047, 0.8594, 0.9063,
    0.9453, 0.9531, 0.9609, 0.9688, 1.0000
])

ghia_v = {
    100: np.array([
         0.00000,  0.09233,  0.10091,  0.10890,  0.12317,  0.16077,
         0.17507,  0.17527,  0.05454, -0.24533, -0.22445, -0.16914,
        -0.10313, -0.08864, -0.07391, -0.05906,  0.00000]),
    400: np.array([
         0.00000,  0.18360,  0.19713,  0.20920,  0.22986,  0.28003,
         0.30174,  0.30203,  0.05186, -0.38598, -0.44993, -0.38829,
        -0.23827, -0.22847, -0.19254, -0.15663,  0.00000]),
    1000: np.array([
         0.00000,  0.27485,  0.29012,  0.30353,  0.32627,  0.37095,
         0.33075,  0.32235,  0.02526, -0.31966, -0.42665, -0.51550,
        -0.39188, -0.33714, -0.27669, -0.21388,  0.00000]),
}


## 9. Run Simulations

> **Runtime (approximate, modern laptop):**  Re=100 ≈ 1 min · Re=400 ≈ 3 min · Re=1000 ≈ 8 min


In [ ]:
results = {}

results[100]  = simulate_cavity_flow(Re=100,  nsteps=11000, dt=0.001)
results[400]  = simulate_cavity_flow(Re=400,  nsteps=30000, dt=0.001)
results[1000] = simulate_cavity_flow(Re=1000, nsteps=50000, dt=0.0005)


## 10. Validation Against Ghia et al.

The computed centreline profiles are compared with the Ghia tabulated data. Accuracy is quantified with the **root-mean-square error (RMSE)**:

$$\text{RMSE} = \sqrt{\frac{1}{N}\sum_{k=1}^{N}\left(\phi_k^{\text{computed}} - \phi_k^{\text{Ghia}}\right)^2}$$

Interpolation to the exact Ghia sample positions is done with `numpy.interp`.


In [ ]:
x_grid = np.linspace(0, 1, nx)
y_grid = np.linspace(0, 1, ny)

print(f"{'Re':>5} | {'RMSE u':>8} | {'RMSE v':>8}")
print(f"{'':-^5}-+-{'':-^8}-+-{'':-^8}")

rmse_table = {}
for Re in [100, 400, 1000]:
    u, v, p = results[Re]

    # Centreline profiles
    u_vert  = u[:, nx // 2]   # u along x = 0.5  (all y)
    v_horiz = v[ny // 2, :]   # v along y = 0.5  (all x)

    # Interpolate at Ghia node locations
    u_interp = np.interp(ghia_y, y_grid, u_vert)
    v_interp = np.interp(ghia_x, x_grid, v_horiz)

    rmse_u = np.sqrt(np.mean((u_interp - ghia_u[Re])**2))
    rmse_v = np.sqrt(np.mean((v_interp - ghia_v[Re])**2))
    rmse_table[Re] = (rmse_u, rmse_v)

    print(f"{Re:>5} | {rmse_u:>8.4f} | {rmse_v:>8.4f}")


### Centreline Velocity Profiles

Computed lines are drawn continuously; Ghia data points are shown as circles.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = {100: 'C0', 400: 'C1', 1000: 'C2'}

# ── Left panel: u at x = 0.5 ────────────────────────────────────────────────
ax = axes[0]
for Re in [100, 400, 1000]:
    u, v, p = results[Re]
    c = colors[Re]
    ax.plot(u[:, nx // 2], y_grid, '-', color=c, lw=1.8,
            label=f'Computed Re={Re}')
    ax.plot(ghia_u[Re], ghia_y, 'o', color=c, ms=5, mec='k', mew=0.5,
            label=f'Ghia et al. Re={Re}')
ax.axvline(0, color='grey', lw=0.5, ls='--')
ax.set_xlabel('$u$ velocity', fontsize=12)
ax.set_ylabel('$y$', fontsize=12)
ax.set_title('$u$-velocity at vertical centreline ($x = 0.5$)', fontsize=11)
ax.legend(fontsize=7, ncol=2)
ax.grid(True, ls='--', alpha=0.35)

# ── Right panel: v at y = 0.5 ────────────────────────────────────────────────
ax = axes[1]
for Re in [100, 400, 1000]:
    u, v, p = results[Re]
    c = colors[Re]
    ax.plot(x_grid, v[ny // 2, :], '-', color=c, lw=1.8,
            label=f'Computed Re={Re}')
    ax.plot(ghia_x, ghia_v[Re], 'o', color=c, ms=5, mec='k', mew=0.5,
            label=f'Ghia et al. Re={Re}')
ax.axhline(0, color='grey', lw=0.5, ls='--')
ax.set_xlabel('$x$', fontsize=12)
ax.set_ylabel('$v$ velocity', fontsize=12)
ax.set_title('$v$-velocity at horizontal centreline ($y = 0.5$)', fontsize=11)
ax.legend(fontsize=7, ncol=2)
ax.grid(True, ls='--', alpha=0.35)

plt.tight_layout()
plt.savefig('validation_centerline.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved as validation_centerline.png")


## 11. Streamline Visualisation

Colour-coded by local flow speed $|\mathbf{u}| = \sqrt{u^2 + v^2}$.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, Re in zip(axes, [100, 400, 1000]):
    u, v, p = results[Re]
    speed = np.sqrt(u**2 + v**2)

    strm = ax.streamplot(
        X, Y, u, v,
        color=speed, linewidth=1.2,
        cmap='plasma', density=1.5,
        arrowsize=0.8
    )
    plt.colorbar(strm.lines, ax=ax, label='Speed  $|\\mathbf{u}|$', shrink=0.9)
    ax.set_title(f'Re = {Re}', fontsize=12)
    ax.set_xlabel('$x$'); ax.set_ylabel('$y$')
    ax.set_aspect('equal')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)

plt.suptitle('Lid-Driven Cavity — Streamlines', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('streamlines.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved as streamlines.png")


## 12. Pressure Field


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, Re in zip(axes, [100, 400, 1000]):
    u, v, p = results[Re]
    vmax = np.abs(p).max()
    im = ax.contourf(X, Y, p, levels=40, cmap='RdBu_r',
                     vmin=-vmax, vmax=vmax)
    plt.colorbar(im, ax=ax, label='$p$', shrink=0.9)
    ax.contour(X, Y, p, levels=20, colors='k', linewidths=0.3, alpha=0.4)
    ax.set_title(f'Pressure — Re = {Re}', fontsize=11)
    ax.set_xlabel('$x$'); ax.set_ylabel('$y$')
    ax.set_aspect('equal')

plt.tight_layout()
plt.savefig('pressure_field.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved as pressure_field.png")


## 13. Discussion

### RMSE Summary

| Re | RMSE $u$ | RMSE $v$ |
|----|----------|----------|
| 100  | 0.0146 | 0.0117 |
| 400  | 0.0255 | 0.0518 |
| 1000 | 0.0637 | 0.0743 |

### Observations

* **Re = 100** matches Ghia closely. The flow is dominated by a single large primary vortex centred near the middle of the cavity.
* **Re = 400** shows noticeably larger error in $v$. Secondary corner vortices begin to form and require better resolution.
* **Re = 1000** has the highest error. At this Reynolds number the corner vortices are prominent and the $65\times65$ grid is near its resolution limit. A finer mesh or a higher-order scheme would improve agreement.

### Known Limitations

* **Fixed Poisson iteration count.** 200 Jacobi iterations per step work well for this grid but convergence is not explicitly checked. A residual-based stopping criterion would be more robust.
* **First-order time integration.** Forward Euler limits accuracy in time; Adams–Bashforth or Runge–Kutta schemes would improve transient behaviour.
* **No adaptive $\Delta t$.** The time-step sizes were chosen conservatively by hand based on CFL and viscous stability conditions.

### Reference

Ghia, U., Ghia, K. N., & Shin, C. T. (1982). High-Re solutions for incompressible flow using the Navier-Stokes equations and a multigrid method. *Journal of Computational Physics*, **48**(3), 387–411. https://doi.org/10.1016/0021-9991(82)90058-4
